# 🎓 Student Placement Prediction | ML Guide

> **Goal:** Predict whether a student will be **Placed** or **Not Placed** using their academic, technical, and lifestyle features.

---

## 📋 Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load & Explore the Data](#2-load--explore-the-data)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Handle Class Imbalance with SMOTE](#5-handle-class-imbalance-with-smote)
6. [Train the Model](#6-train-the-model)
7. [Evaluate the Model](#7-evaluate-the-model)
8. [Feature Importance](#8-feature-importance)
9. [Conclusion](#9-conclusion)

---


## 1. Import Libraries
We import all the tools we need. Think of this as gathering ingredients before cooking. 🍳

In [ ]:
# ── Data handling ──────────────────────────────────────────
import pandas as pd          # Work with tables (DataFrames)
import numpy as np           # Math operations

# ── Visualisation ───────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

# ── Machine Learning ────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
                              confusion_matrix,
                              roc_auc_score,
                              roc_curve,
                              ConfusionMatrixDisplay)

# ── Imbalanced data handler ─────────────────────────────────
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully!")


## 2. Load & Explore the Data
Let's load the CSV file and take a first look. 👀

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/student-placement-career-success/student_placement_career_success_dataset.csv')

print(f"Dataset Shape: {df.shape}")  # rows × columns
print(f"Columns     : {df.shape[1]}")
df.head()                            # Show first 5 rows


In [ ]:
# Quick summary statistics
df.describe().T.style.background_gradient(cmap='Blues')


In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0].reset_index()
missing.columns = ['Column', 'Missing Count']
missing['Missing %'] = (missing['Missing Count'] / len(df) * 100).round(2)
print(missing)


In [ ]:
# Check the target column distribution
print("Target Column: placement_status")
print(df['placement_status'].value_counts())
print()
# Visualise it
df['placement_status'].value_counts().plot(
    kind='bar', color=['steelblue','tomato'], edgecolor='black'
)
plt.title('Placement Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Exploratory Data Analysis (EDA)
EDA helps us **understand** the data — patterns, relationships, and outliers — before building any model.

In [ ]:
# CGPA distribution by placement status
plt.figure(figsize=(8,4))
sns.histplot(data=df, x='cgpa', hue='placement_status',
             kde=True, bins=30, palette=['tomato','steelblue'])
plt.title('CGPA Distribution by Placement Status')
plt.xlabel('CGPA')
plt.tight_layout()
plt.show()


In [ ]:
# Top features correlation heatmap (numeric only)
num_df = df.select_dtypes(include='number')
corr = num_df.corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.3)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()


In [ ]:
# Internships vs Placement
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='internships_completed',
              hue='placement_status', palette=['tomato','steelblue'])
plt.title('Internships Completed vs Placement')
plt.xlabel('Internships Completed')
plt.tight_layout()
plt.show()


In [ ]:
# College Tier vs Placement
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='college_tier',
              hue='placement_status', palette=['tomato','steelblue'])
plt.title('College Tier vs Placement Status')
plt.xlabel('College Tier')
plt.tight_layout()
plt.show()


## 4. Data Preprocessing
Before feeding data to a model we need to:
1. **Drop** columns that would **leak** target information (salary, offer_count, etc.)
2. **Fill** missing values (imputation)
3. **Encode** text/categorical columns to numbers


In [ ]:
# Step 4a – Drop leaky / post-placement columns
# These columns are only known AFTER placement, so we must remove them.
leaky_cols = [
    'student_id',            # Just an ID
    'salary_lpa',            # Known only after getting placed
    'company_type',          # Known only after getting placed
    'work_mode',             # Known only after getting placed
    'offer_count',           # Known only after getting placed
    'interview_rounds_cleared',  # Known only after getting placed
    'joining_delay_months',  # Known only after getting placed
]
df.drop(columns=leaky_cols, inplace=True)
print(f"Remaining columns: {df.shape[1]}")


In [ ]:
# Step 4b – Encode the target column (text → 0/1)
# Placed = 1,  Not Placed = 0
df['placement_status'] = (df['placement_status'] == 'Placed').astype(int)
print("Target encoding done ✅")
print(df['placement_status'].value_counts())


In [ ]:
# Step 4c – Encode categorical features (text → number)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Label encoding done ✅")


In [ ]:
# Step 4d – Separate features (X) and target (y)
X = df.drop('placement_status', axis=1)
y = df['placement_status']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")


In [ ]:
# Step 4e – Fill missing values with median
# Median is robust to outliers, great for numeric data
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f"Missing values remaining: {X.isnull().sum().sum()} ✅")


In [ ]:
# Step 4f – Train-Test Split (80% train, 20% test)
# stratify=y keeps the same class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Train size : {X_train.shape[0]} rows")
print(f"Test  size : {X_test.shape[0]} rows")


## 5. Handle Class Imbalance with SMOTE
Our dataset has **~24,596 Placed** vs **~404 Not Placed** — heavily imbalanced!

If we ignore this, the model will just predict "Placed" for everyone and look 98% accurate — but it's useless.

**SMOTE** (Synthetic Minority Over-sampling Technique) creates *synthetic* (fake but realistic) examples of the minority class to balance the dataset.


In [ ]:
# Apply SMOTE only on TRAINING data (never on test data!)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", dict(zip(*np.unique(y_train, return_counts=True))))
print("After  SMOTE:", dict(zip(*np.unique(y_train_res, return_counts=True))))


## 6. Train the Model – Random Forest Classifier 🌲
**Random Forest** builds many decision trees and combines their results (majority vote).
- Simple to understand
- Handles mixed data types well
- Provides feature importance
- Robust to overfitting


In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,      # Build 100 trees
    class_weight='balanced', # Extra weight on minority class
    random_state=42
)

rf_model.fit(X_train_res, y_train_res)
print("✅ Model training complete!")


## 7. Evaluate the Model
We use several metrics:
- **Accuracy** – % of correct predictions
- **Precision** – Of all predicted Placed, how many are actually Placed?
- **Recall** – Of all actually Placed, how many did we catch?
- **F1-Score** – Balance between Precision and Recall
- **ROC-AUC** – Overall model quality (closer to 1 is better)


In [ ]:
# Predict on test data
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]  # Probability of 'Placed'

# Classification report
print("=" * 55)
print("         CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Not Placed','Placed']))
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Placed','Placed'])
fig, ax = plt.subplots(figsize=(5,4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='steelblue', lw=2,
         label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0,1],[0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Random Forest')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 8. Feature Importance
Which features matter most to the model?

In [ ]:
# Top 15 important features
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(8,6))
top15.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## 9. Conclusion 🏁

### ✅ What We Did
| Step | Action |
|------|--------|
| EDA | Explored distributions, correlations, and class imbalance |
| Preprocessing | Dropped leaky columns, encoded categoricals, imputed missing values |
| SMOTE | Balanced the minority class synthetically on training data only |
| Model | Trained a Random Forest Classifier with `class_weight='balanced'` |
| Evaluation | Reported Accuracy, F1-Score, and ROC-AUC |

---

### 📊 Model Performance Summary
| Metric | Score |
|--------|-------|
| Accuracy | ~98% |
| ROC-AUC | ~0.89 |
| Weighted F1 | ~0.98 |

---

### 🔑 Key Insights
- **`layoffs_risk_score`**, **`internships_completed`**, and **`cgpa`** are the top 3 predictors of placement.
- Students from **Tier 1 colleges** with **more internships** and **higher CGPA** are significantly more likely to be placed.
- The class imbalance (96% Placed) makes this a challenging problem — SMOTE helps the model learn about the minority class.

---

### 🚀 Next Steps (to improve further)
- Try **XGBoost** or **LightGBM** for potentially better results.
- Use **GridSearchCV** to tune hyperparameters.
- Experiment with different sampling ratios in SMOTE.
- Try **threshold tuning** to improve recall for "Not Placed" students.

---

> ⭐ *If this notebook helped you, please give it an upvote! It motivates me to create more beginner-friendly content.*
